In [223]:
import pandas as pd
import numpy as np

In [224]:
df = pd.read_csv("/content/flats - flats.csv")

In [278]:

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [225]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3028 entries, 0 to 3027
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   property_name    3028 non-null   object
 1   link             3028 non-null   object
 2   society          3027 non-null   object
 3   price            3008 non-null   object
 4   area             3015 non-null   object
 5   areaWithType     3009 non-null   object
 6   bedRoom          3009 non-null   object
 7   bathroom         3009 non-null   object
 8   balcony          3009 non-null   object
 9   additionalRoom   1695 non-null   object
 10  address          3003 non-null   object
 11  floorNum         3007 non-null   object
 12  facing           2128 non-null   object
 13  agePossession    3008 non-null   object
 14  nearbyLocations  2914 non-null   object
 15  description      3009 non-null   object
 16  furnishDetails   2204 non-null   object
 17  features         2595 non-null   

In [226]:
df = df[(df['price'].notna())]

In [227]:
df =  df[df['property_name'] !='property_name']

In [228]:
df.duplicated().sum()

np.int64(0)

In [229]:
df.isna().sum()

,0
property_name,0
link,0
society,1
price,0
area,11
areaWithType,0
bedRoom,0
bathroom,0
balcony,0
additionalRoom,1313


In [230]:
df.drop('link', axis=1, inplace=True)

In [231]:
df.rename(columns = {'area':'price_per_sqft'},inplace = True)

In [232]:
import re
df['society'] = df['society'].apply(lambda name: re.sub(r'\d+(\.\d+)?\s?★', '', str(name)).strip()).str.lower()

In [233]:
# Keep everything except "Price on Request"
df = df[df['price'] != "Price on Request"]


In [234]:
def treat_price(x):
  if type(x)==float:
    return x
  else:
    if x[1] =='Lac':
      return round(float(x[0])/100,2)
    else:
      return round(float(x[0]),2)

In [235]:
df['price'] = df['price'].str.split(" ").apply(treat_price)


In [236]:
df[df['price_per_sqft'].isna()]

,property_name,society,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating,property_id


In [237]:
def treat_price_per_sqft(x):
  return round(float(x[1]),2)

In [239]:
df['price_per_sqft']=df['price_per_sqft'].str.replace('/', ' ').str.replace(',','').str.strip().str.split().apply(treat_price_per_sqft)

In [250]:
df['bedRoom']=df['bedRoom'].str.split(" ").str.get(0).astype('int')

In [251]:
df['bathroom']=df['bathroom'].str.split(" ").str.get(0).astype('int')

In [263]:
df['balcony']=df['balcony'].str.split(" ").str.get(0).str.replace('No','0')

In [265]:
df['additionalRoom'].isna().sum()

np.int64(1304)

In [270]:
df['additionalRoom'].fillna('Not available',inplace=True)

In [273]:
df['additionalRoom'] = df['additionalRoom'].str.lower()

In [290]:
#Extracting floor no from the floornum column
df['floorNum']=df['floorNum'].str.replace('Ground','0').str.replace('Basement','-1').str.replace('Lower','0').str.split().str.get(0).str.extract(r'(-?\d+)')

In [292]:
df['facing'].fillna('NA',inplace = True)

/tmp/ipykernel_1581/774203886.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['facing'].fillna('NA',inplace = True)


In [294]:
# Add a new column to calculate Total Area
df.insert(
    loc=4,
    column='area',
    value=((df['price'] * 10000000) / df['price_per_sqft']).round()
)

In [298]:
df.insert(
    loc = 1,
    column = 'property_type',
    value = 'flat'
    )

In [300]:
df.head(1)

,property_name,property_type,society,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,address,floorNum,facing,agePossession,nearbyLocations,description,furnishDetails,features,rating,property_id
0,2 BHK Flat in Krishna Colony,flat,maa bhagwati residency,0.45,5000.0,900.0,Carpet area: 900 (83.61 sq.m.),2,2,1,not available,"Krishna Colony, Gurgaon, Haryana",4,West,1 to 5 Year Old,"['Chintapurni Mandir', 'State bank ATM', 'Pear...",So with lift.Maa bhagwati residency is one of ...,"['3 Fan', '4 Light', '1 Wardrobe', 'No AC', 'N...","['Feng Shui / Vaastu Compliant', 'Security / F...","['Environment4 out of 5', 'Safety4 out of 5', ...",C68850746


In [302]:
df.to_csv('flats_cleaned.csv',index=False)

In [303]:
df.shape

(2996, 21)